# 条件筛选与简单整数索引

学习目标：按条件或指定位置选择数组数据，区分筛选与保形选择，并用逻辑归约检查条件。

前置知识：数组比较、基本索引、数组形状与轴、布尔逻辑。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的模拟数据，后续单元沿用首次导入的 np。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 按条件筛选

筛选达到 20.0 °C 的温度，可以先比较，再把比较结果放入方括号。比较得到的布尔数组称为布尔掩码（Boolean mask）：True 表示选取，False 表示跳过。

下面读数和掩码的形状均为 (5,)，筛选结果只含满足条件的值，并保持它们在原数组中的先后顺序。

In [1]:
import numpy as np

temperatures = np.array([18.0, 20.0, 22.5, 19.0, 24.0])
mask = temperatures >= 20.0
selected = temperatures[mask]

print(mask)  # 预期：[False True True False True]。
print(mask.shape, mask.dtype)  # 预期：(5,) bool。
print(selected)  # 预期：[20. 22.5 24.]，包含恰好达到阈值的 20.0。
print(selected.shape, selected.dtype)  # 预期：(3,) float64。

[False  True  True False  True]
(5,) bool
[20.  22.5 24. ]
(3,) float64


## 2 组合条件

### 2.1 逻辑运算与括号

筛选同时满足多个条件的值，可以对布尔掩码使用 &；至少满足一个条件用 |，取反用 ~。每个比较式分别加括号，因为 & 和 | 的优先级高于比较运算。

下面使用温度数组筛选 20.0～23.0 °C 的值，包含两端。

In [2]:
temperatures = np.array([18.0, 20.0, 22.5, 23.0, 24.0])
within_range = (temperatures >= 20.0) & (temperatures <= 23.0)
outside_range = (temperatures < 20.0) | (temperatures > 23.0)

print(temperatures[within_range])  # 预期：[20. 22.5 23.]。
print(outside_range)  # 预期：[True False False False True]。
print(~within_range)  # 与 outside_range 相同，仍为形状 (5,) 的 bool 数组。

[20.  22.5 23. ]
[ True False False False  True]
[ True False False False  True]


也可以使用逻辑函数表达相同条件。下表中的 left、right 表示同形布尔数组，mask 表示一个布尔数组。

| 函数 | 中文名称／含义 | 对布尔数组的等价写法 |
| --- | --- | --- |
| np.logical_and(left, right) | 逐元素逻辑与，两者均为真 | left & right |
| np.logical_or(left, right) | 逐元素逻辑或，至少一者为真 | left \| right |
| np.logical_not(mask) | 逐元素逻辑非 | ~mask |

函数接收完整的比较表达式，无需借助运算符优先级组合条件。

In [3]:
temperatures = np.array([18.0, 20.0, 22.5, 23.0, 24.0])
within_range = np.logical_and(temperatures >= 20.0, temperatures <= 23.0)
outside_range = np.logical_or(temperatures < 20.0, temperatures > 23.0)

print(within_range)  # 预期：[False True True True False]。
print(outside_range)  # 预期：[True False False False True]。
print(np.logical_not(within_range))  # 与 outside_range 相同。

[False  True  True  True False]
[ True False False False  True]
[ True False False False  True]


### 2.2 整数位运算与布尔逻辑

& 本身是按位与运算符。作用于布尔数组时可以组合条件；作用于整数数组时，计算的是整数的二进制位。np.logical_and() 则判断各元素的真值，数值零为假，非零为真。

不要用整数数组中的 0、1 替代布尔掩码：整数数组放进方括号会被当作位置索引。浮点数组也不能直接做按位与。

In [4]:
left = np.array([2, 3], dtype=np.int64)
right = np.array([1, 1], dtype=np.int64)

print(left & right)  # 预期：[0 1]；二进制 10 & 01 为 00，11 & 01 为 01。
print(np.logical_and(left, right))  # 预期：[True True]，两组元素均非零。

values = np.array([10, 20, 30])
print(values[np.array([0, 1])])  # 预期：[10 20]，0、1 是位置。
print(values[np.array([False, True, False])])  # 预期：[20]，按布尔条件筛选。

[0 1]
[ True  True]
[10 20]
[20]


### 2.3 数组的整体真值

Python 的 and、or、not，以及 if 条件，需要判断对象的整体真值，不能代替数组的逐元素逻辑。NumPy 2.5 中，元素数量不为 1 的数组不能直接作整体真值判断；判断是否为空用 size。

下面的反例直接显示 ValueError。需要判断“是否存在”或“是否全部满足”时，使用第 6 节的 any() 或 all()。

In [5]:
values = np.array([1, 2, 3])

# 预期 ValueError：and 需要判断整个数组的真值，多元素布尔数组不能这样判断。
invalid_mask = (values >= 2) and (values <= 3)

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [6]:
valid_mask = (values >= 2) & (values <= 3)
print(valid_mask)  # 预期：[False True True]。

[False  True  True]


## 3 按指定位置选取

整数列表或整数数组可以按指定顺序选取数据。对于一维输入，每个整数指定一个元素；对于二维输入，单独放入方括号的整数序列指定行。负数位置仍从轴的末尾计数。

下面 rows 的形状为 (3, 2)，行表示三次观测，列表示两个传感器。整数序列 [2, 0] 先取第三行，再取第一行。

In [7]:
rows = np.array([[10, 11], [20, 21], [30, 31]], dtype=np.int64)
positions = np.array([2, 0])
selected_rows = rows[positions]

print(selected_rows)  # 预期：先 [30 31]，后 [10 11]。
print(selected_rows.shape, selected_rows.dtype)  # 预期：(2, 2) int64。
print(rows[[-1, 0]])  # 与 selected_rows 相同，-1 指最后一行。
print(rows[[1]].shape)  # 预期：(1, 2)，整数列表选取仍保留行轴。
print(rows[1].shape)  # 预期：(2,)，单个整数索引移除行轴。

[[30 31]
 [10 11]]
(2, 2) int64
[[30 31]
 [10 11]]
(1, 2)
(2,)


整数索引必须落在对应轴的合法范围内。上面的行轴长度为 3，非负行位置只能是 0、1、2。

没有要选的位置时，可以使用空整数数组；其结果仍保留未选择轴的形状。

In [8]:
rows = np.array([[10, 11], [20, 21], [30, 31]], dtype=np.int64)
empty_positions = np.array([], dtype=np.int64)
empty_rows = rows[empty_positions]
print(empty_rows.shape, empty_rows.dtype)  # 预期：(0, 2) int64，没有行但仍有两列。

# 预期 IndexError：行轴只有位置 0、1、2，索引 3 越界。
rows[[3]]

(0, 2) int64


IndexError: index 3 is out of bounds for axis 0 with size 3

## 4 多维布尔掩码

### 4.1 选择元素与选择整行

与整个二维数组同形的掩码按元素筛选，结果压成一维，按逐行顺序收集 True 位置的值。

下面 temperatures 的形状为 (3, 2)，行表示观测，列表示传感器，数值单位为摄氏度。

In [9]:
temperatures = np.array([[18.0, 21.0], [22.0, 19.0], [24.0, 23.0]])
element_mask = temperatures >= 21.0
selected = temperatures[element_mask]

print(element_mask)  # 预期：三行依次为 [False True]、[True False]、[True True]。
print(element_mask.shape)  # 预期：(3, 2)，与原数组同形。
print(selected)  # 预期：[21. 22. 24. 23.]，按逐行顺序收集。
print(selected.shape, selected.dtype)  # 预期：(4,) float64。

[[False  True]
 [ True False]
 [ True  True]]
(3, 2)
[21. 22. 24. 23.]
(4,) float64


如果要保留整条观测，使用长度等于行数的一维布尔掩码。每个 True 选中一整行，结果保留列轴。

下面根据第一个传感器的温度决定是否保留该次观测；继续使用上一单元的 temperatures。

In [10]:
row_mask = temperatures[:, 0] >= 21.0
selected_rows = temperatures[row_mask]

print(row_mask)  # 预期：[False True True]，形状为 (3,)。
print(selected_rows)  # 预期：保留 [22. 19.]、[24. 23.] 两行。
print(selected_rows.shape, selected_rows.dtype)  # 预期：(2, 2) float64。

[False  True  True]
[[22. 19.]
 [24. 23.]]
(2, 2) float64


### 4.2 空结果与形状错误

掩码全为 False 时得到空结果。元素掩码得到形状 (0,)；行掩码得到零行、列数不变的二维数组。

布尔掩码的长度或形状必须与它选择的区域对应，不能只看 True 的数量。

In [11]:
temperatures = np.array([[18.0, 21.0], [22.0, 19.0], [24.0, 23.0]])
empty_values = temperatures[temperatures > 100.0]
empty_rows = temperatures[temperatures[:, 0] > 100.0]

print(empty_values.shape, empty_values.dtype)  # 预期：(0,) float64。
print(empty_rows.shape, empty_rows.dtype)  # 预期：(0, 2) float64。
print(empty_values.size == 0)  # 预期：True，明确检查是否为空。

wrong_mask = np.array([True, False])

# 预期 IndexError：两项布尔掩码与三行输入不匹配。
temperatures[wrong_mask]

(0,) float64
(0, 2) float64
True


IndexError: boolean index did not match indexed array along axis 0; size of axis is 3 but size of corresponding boolean axis is 2

## 5 条件选择与位置查找

### 5.1 where 保留对应位置

需要替换不满足条件的值，同时保留原来的行列结构时，使用 np.where(condition, x, y)：condition 为真时取 x，为假时取 y。

condition 是布尔条件，x、y 是两种候选值。下面条件和温度数组同形，另一个候选是标量 -99.0，表示本例约定的占位值。结果保持 (3, 2)，与方括号筛选得到的一维结果不同。

In [12]:
temperatures = np.array([[18.0, 21.0], [22.0, 19.0], [24.0, 23.0]])
mask = temperatures >= 21.0
marked = np.where(mask, temperatures, -99.0)

print(marked)  # 18.0 和 19.0 所在位置变为 -99.0，其余位置保留原值。
print(marked.shape, marked.dtype)  # 预期：(3, 2) float64。
print(temperatures[mask].shape)  # 预期：(4,)，方括号只收集满足条件的值。
print(temperatures)  # 原数组没有被 where 修改。

[[-99.  21.]
 [ 22. -99.]
 [ 24.  23.]]
(3, 2) float64
(4,)
[[18. 21.]
 [22. 19.]
 [24. 23.]]


### 5.2 nonzero 查找位置

np.nonzero() 返回非零元素的位置；传入布尔数组时，返回 True 的位置。返回值是元组，每个轴对应一个整数数组，顺序为逐行扫描顺序。

对二维条件，返回的第一个数组是行位置，第二个是列位置；同一序号处的两个整数共同表示一个坐标。只传条件的 np.where(condition) 也返回位置，查找位置时优先直接使用 nonzero()。

In [13]:
temperatures = np.array([[18.0, 21.0], [22.0, 19.0], [24.0, 23.0]])
row_positions, column_positions = np.nonzero(temperatures >= 22.0)

print(row_positions)  # 预期：[1 2 2]。
print(column_positions)  # 预期：[0 0 1]，坐标依次为 (1, 0)、(2, 0)、(2, 1)。
print(row_positions.shape, column_positions.shape)  # 均为 (3,)，共三个位置。
print(np.nonzero(temperatures > 100.0))  # 元组中有两个空整数数组。

[1 2 2]
[0 0 1]
(3,) (3,)
(array([], dtype=int64), array([], dtype=int64))


## 6 any 与 all

### 6.1 检查整体条件

np.any() 判断是否至少有一个元素为真，np.all() 判断是否全部为真。不指定 axis 时检查整个数组，返回单个布尔结果，可用于 if。

下面检查温度是否达到 21.0 °C。先写出比较条件，再归约，避免把“数值非零”误当作“达到阈值”。

In [14]:
temperatures = np.array([[18.0, 21.0], [22.0, 19.0], [24.0, 23.0]])
mask = temperatures >= 21.0

print(np.any(mask))  # 预期：True，至少有一个位置达到阈值。
print(np.all(mask))  # 预期：False，并非每个位置都达到阈值。
if np.any(mask):
    print("存在达到阈值的读数")  # 条件是归约后的单个布尔结果。

True
False
存在达到阈值的读数


### 6.2 按轴检查条件

对于形状 (3, 2) 的布尔数组，axis=1 合并每行的两列条件，留下三个行结果；axis=0 合并每列的三行条件，留下两个列结果。

下面仍约定行表示观测、列表示传感器。按行得到的布尔结果可以直接用来选择整条观测。

In [15]:
temperatures = np.array([[18.0, 21.0], [22.0, 19.0], [24.0, 23.0]])
mask = temperatures >= 21.0
any_in_row = np.any(mask, axis=1)
all_in_row = np.all(mask, axis=1)
all_in_column = np.all(mask, axis=0)

print(any_in_row, any_in_row.shape)  # 预期：[True True True] (3,)。
print(all_in_row, all_in_row.shape)  # 预期：[False False True] (3,)。
print(all_in_column, all_in_column.shape)  # 预期：[False False] (2,)。
print(temperatures[all_in_row])  # 预期：仅保留 [24. 23.]，形状为 (1, 2)。

[ True  True  True] (3,)
[False False  True] (3,)
[False False] (2,)
[[24. 23.]]


## 7 综合应用：选择完整观测

三个时刻、两个传感器的模拟温度如下。任务要求每条观测的所有读数都在 18.0～24.0 °C 内，包含两端；只保留符合条件的整行，再按指定顺序输出。

先构造元素条件，再沿列轴归约成行条件。指定顺序中的位置相对于筛选后的数组。

In [16]:
temperatures = np.array([[18.0, 22.0], [25.0, 20.0], [21.0, 24.0]])
valid_elements = (temperatures >= 18.0) & (temperatures <= 24.0)
valid_rows = np.all(valid_elements, axis=1)
accepted = temperatures[valid_rows]
ordered = accepted[[1, 0]]

print(valid_rows)  # 预期：[True False True]，第二条观测含超范围的值。
print(accepted)  # 预期：[18. 22.]、[21. 24.]，形状为 (2, 2)。
print(ordered)  # 预期：先 [21. 24.]，后 [18. 22.]。
print(ordered.shape, ordered.dtype)  # 预期：(2, 2) float64。

[ True False  True]
[[18. 22.]
 [21. 24.]]
[[21. 24.]
 [18. 22.]]
(2, 2) float64


## 8 选学：多分支选择与按轴选取

### 8.1 select 的条件顺序

多种条件对应不同结果时，可以使用 np.select()。第一个参数列出条件，第二个参数按相同顺序列出候选值，default 指定全部条件均为假时的值。多个条件同时为真时，采用排在最前面的条件。

下面用整数代码表示模拟温度等级：2 表示达到 30.0 °C，1 表示达到 20.0 °C，0 表示低于 20.0 °C。

In [17]:
temperatures = np.array([18.0, 20.0, 30.0])
levels = np.select([temperatures >= 30.0, temperatures >= 20.0], [2, 1], default=0)

print(levels)  # 预期：[0 1 2]；30.0 满足两个条件，采用前面的等级 2。
print(levels.shape)  # 预期：(3,)，每个输入位置都有对应结果。

[0 1 2]
(3,)


### 8.2 take 指定轴

np.take() 用 axis 明确指定沿哪个轴取值，适合按同一位置序列选行或选列。不指定 axis 时，先把输入看作一维序列；需要按行或列选取时应显式指定轴。

下面数组的形状为 (3, 2)，行表示观测、列表示传感器。

In [18]:
rows = np.array([[10, 11], [20, 21], [30, 31]], dtype=np.int64)
selected_rows = np.take(rows, [2, 0], axis=0)
selected_columns = np.take(rows, [1, 0], axis=1)

print(selected_rows)  # 先第三行再第一行，与 rows[[2, 0]] 相同。
print(selected_rows.shape)  # 预期：(2, 2)。
print(selected_columns)  # 每行先第二列再第一列。
print(selected_columns.shape)  # 预期：(3, 2)。
print(np.take(rows, [2, 0]))  # 预期：[20 10]，省略 axis 后位置指向展平序列。

[[30 31]
 [10 11]]
(2, 2)
[[11 10]
 [21 20]
 [31 30]]
(3, 2)
[20 10]


## 本章小结

（1）布尔掩码选 True 对应的数据，整数序列按指定位置与顺序选取；两者不能互换。

（2）同形元素掩码把选中值收集为一维数组，行掩码保留未选择的列轴。全 False 可以产生空结果，形状仍取决于选择范围。

（3）组合比较条件时，对每个比较式加括号，再使用 &、|、~ 或逻辑函数。数组整体真值不能代替逐元素逻辑。

（4）where() 在两种候选值之间选择，nonzero() 返回位置；any() 和 all() 用于整体或按轴检查条件。

（5）运行前应能说明掩码选择哪个区域、结果是否保留行列结构，以及整数位置相对于哪一个数组。

## 练习

（1）筛选 18.0～22.0 °C 的温度，包含两端；再筛选高于 100.0 °C 的值，分别检查值、形状和类型。

In [19]:
temperatures = np.array([17.0, 18.0, 20.0, 22.0, 23.0])

# 在此编写区间条件与筛选代码。
# 检查：区间筛选得到 [18. 20. 22.]，shape 为 (3,)，dtype 为 float64。
# 检查：高于 100.0 的筛选结果 shape 为 (0,)，dtype 仍为 float64。

（2）先预测下面两个选择结果的值和形状，再运行。说明两个掩码分别作用于什么范围。

In [20]:
values = np.array([[1, 5], [6, 2], [7, 8]])
element_result = values[values >= 5]
row_result = values[values[:, 0] >= 5]

# 先记录预测，再核对元素数量、行列结构和选取顺序。
print(element_result, element_result.shape)
print(row_result, row_result.shape)

[5 6 7 8] (4,)
[[6 2]
 [7 8]] (2, 2)


（3）输出必须保持三个时刻、两个传感器的位置关系，把低于 20.0 °C 的读数改为 -99.0，其余值保持不变。选择布尔筛选或 where() 完成任务，在注释中解释选择理由；再说明若要求只保留合格值，方法应如何变化。

In [21]:
temperatures = np.array([[18.0, 22.0], [21.0, 19.0], [23.0, 24.0]])

# 在此完成保形选择并说明理由。
# 检查：结果 shape 为 (3, 2)，只有 (0, 0)、(1, 1) 位置变为 -99.0。
# 检查：原数组不变，结果 dtype 为 float64。
# 条件改为“只保留合格值”时，说明结果为什么不再需要三行两列。

（4）从下面四条观测中保留“两个传感器均不超过 24.0 °C”的整行，再按筛选结果的位置 [2, 0] 选取。打印不合格读数在原数组中的行、列位置。

In [22]:
temperatures = np.array([[18.0, 22.0], [25.0, 20.0], [21.0, 24.0], [23.0, 19.0]])

# 在此用 all() 生成行条件，筛选后按整数序列重新选取。
# 检查：行条件为 [True False True True]，筛选结果 shape 为 (3, 2)。
# 检查：最后先输出 [23. 19.]，再输出 [18. 22.]，shape 为 (2, 2)。
# 用 nonzero() 检查原输入：不合格读数仅位于 (1, 0)。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | 索引：[Indexing on ndarrays](https://numpy.org/doc/2.5/user/basics.indexing.html) 的 Integer array indexing、Boolean array indexing：单轴整数序列、负位置与越界、同形和低维布尔掩码的顺序与形状条件。条件与位置：[where](https://numpy.org/doc/2.5/reference/generated/numpy.where.html) 的 condition、x、y、Returns 与单参数 Note；[nonzero](https://numpy.org/doc/2.5/reference/generated/numpy.nonzero.html) 的逐轴返回元组与逐行顺序。逻辑：[logical_and](https://numpy.org/doc/2.5/reference/generated/numpy.logical_and.html)、[logical_or](https://numpy.org/doc/2.5/reference/generated/numpy.logical_or.html)、[logical_not](https://numpy.org/doc/2.5/reference/generated/numpy.logical_not.html) 的逐元素真值和布尔运算符示例；[bitwise_and](https://numpy.org/doc/2.5/reference/generated/numpy.bitwise_and.html) 的整数二进制位语义与输入类型；[ndarray](https://numpy.org/doc/2.5/reference/arrays.ndarray.html#arithmetic-matrix-multiplication-and-comparison-operations) 的 Truth value of an array。归约：[any](https://numpy.org/doc/2.5/reference/generated/numpy.any.html)、[all](https://numpy.org/doc/2.5/reference/generated/numpy.all.html) 的 axis、Returns 与按轴示例。选学：[select](https://numpy.org/doc/2.5/reference/generated/numpy.select.html) 的 condlist 顺序与 default；[take](https://numpy.org/doc/2.5/reference/generated/numpy.take.html) 的 axis、indices 与默认展平输入。 |
| Python 官方文档（Python 3.12） | [Expressions — Boolean operations](https://docs.python.org/3.12/reference/expressions.html#boolean-operations) 的 and、or、not 真值判断；[Operator precedence](https://docs.python.org/3.12/reference/expressions.html#operator-precedence) 的位运算与比较优先级。 |